In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Probabilistic Chain Rule Pipeline Benchmark (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook combines all specialized models to calculate **individual probabilities for each of the 5 ESI classes ($P(\text{ESI 1}), P(\text{ESI 2}), P(\text{ESI 3}), P(\text{ESI 4}), P(\text{ESI 5})$)** using the **Probability Chain Rule / Joint Probability Product** on the **15% Holdout Test Set**:

### Models Integrated
1. **Model 1 ([`models/xgboost_raw_esi1_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/xgboost_raw_esi1_extreme.ipynb))**: Binary XGBoost ESI 1 Detector $\rightarrow$ predicts $P_1(\text{ESI 1})$.
2. **Model 2 ([`models/rf_esi23_esi45_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/rf_esi23_esi45_extreme.ipynb))**: Random Forest / XGBoost Router $\rightarrow$ predicts macro branch probability $P_{\text{router}}(\text{ESI 2/3} \mid \text{Not ESI 1})$.
3. **Model 3A ([`models/xgboost_esi23_esi45_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/xgboost_esi23_esi45_extreme.ipynb))**: LightGBM ESI 2 vs 3 Specialist $\rightarrow$ predicts $P_{\text{LGB}}(\text{ESI 2} \mid \text{ESI } 2 \text{ or } 3)$.
4. **Model 3B ([`models/xgboost_esi23_esi45_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/xgboost_esi23_esi45_extreme.ipynb))**: XGBoost ESI 4 vs 5 Specialist $\rightarrow$ predicts $P_{\text{XGB}}(\text{ESI 4} \mid \text{ESI } 4 \text{ or } 5)$.

### Individual 5-Class Probability Chain Rule Formulas
$$\begin{aligned}
P(\text{ESI 1}) &= P_1(\text{ESI 1}) \\[6pt]
P(\text{ESI 2}) &= (1 - P_1(\text{ESI 1})) \times P_{\text{router}}(\text{ESI 2/3}) \times P_{\text{LGB}}(\text{ESI 2} \mid 2, 3) \\[6pt]
P(\text{ESI 3}) &= (1 - P_1(\text{ESI 1})) \times P_{\text{router}}(\text{ESI 2/3}) \times (1 - P_{\text{LGB}}(\text{ESI 2} \mid 2, 3)) \\[6pt]
P(\text{ESI 4}) &= (1 - P_1(\text{ESI 1})) \times P_{\text{router}}(\text{ESI 4/5}) \times P_{\text{XGB}}(\text{ESI 4} \mid 4, 5) \\[6pt]
P(\text{ESI 5}) &= (1 - P_1(\text{ESI 1})) \times P_{\text{router}}(\text{ESI 4/5}) \times (1 - P_{\text{XGB}}(\text{ESI 4} \mid 4, 5))
\end{aligned}$$

*Proof of Total Probability*: $\sum_{k=1}^{5} P(\text{ESI } k) = 1.0$ (100% normalized across all 5 classes).

### 5-Class Evaluation Suite
Evaluates full 5x5 Confusion Matrix, Accuracy, **Balanced Accuracy**, **Specificity**, Precision, Recall/Sensitivity, F1 Score, ROC-AUC, and **MCC Score** across ESI classes 1, 2, 3, 4, and 5.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(ranger)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== 5-Class Chain Rule Pipeline Benchmark Initialized ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Dataset & Construct 45 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
# Stratified 15% Holdout Test Partitioning
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
test_df      <- df_full[-in_train_val, ]
cat(sprintf("Holdout Test Set Ready: %d rows\n", nrow(test_df)))
cat("Natural 5-Class Target Distribution (ESI 1 to 5):\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load Saved Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
path_esi1 <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
path_rf   <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
path_ds   <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
path_lgb23 <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
path_xgb45 <- file.path(deploy_dir, "xgboost_esi45_model.rds")
cat("Loading model artifacts...\n")
art_esi1 <- readRDS(path_esi1)
art_rf   <- readRDS(path_rf)
art_ds   <- readRDS(path_ds)
art_lgb23 <- if (file.exists(path_lgb23)) readRDS(path_lgb23) else NULL
art_xgb45 <- if (file.exists(path_xgb45)) readRDS(path_xgb45) else NULL
cat("  - Model 1 (XGBoost ESI 1 Detector) Loaded.\n")
cat("  - Model 2 (Random Forest / XGBoost Router) Loaded.\n")
cat("  - Model 3A (LightGBM ESI 2 vs 3 Specialist) Loaded.\n")
cat("  - Model 3B (XGBoost ESI 4 vs 5 Specialist) Loaded.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Calculate Individual 5-Class Probabilities via Chain Rule
# ---------------------------------------------------------
# 1. Model 1: P(ESI 1)
raw_feats_16 <- c("age", "gender", "cc_breathingdifficulty", "triage_vital_hr", "triage_vital_sbp", 
                  "triage_vital_rr", "triage_vital_o2", "pulse_last", "resp_last", "spo2_last", 
                  "sbp_last", "resp_min", "spo2_min", "sbp_min", "resp_max", "sbp_max")
test_esi1_scaled <- predict(art_esi1$preproc, test_df[, raw_feats_16])
dtest_esi1       <- xgb.DMatrix(data = as.matrix(test_esi1_scaled))
p_esi1           <- predict(art_esi1$model, dtest_esi1)
p_not_esi1       <- 1 - p_esi1
# 2. Model 2: Router P(ESI 2/3 | Not ESI 1) & P(ESI 4/5 | Not ESI 1)
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols   <- setdiff(names(test_df), c(binary_cols, "target_col"))
test_rf_scaled <- predict(art_rf$preproc, test_df)
rf_raw_probs   <- predict(art_rf$model, data = test_rf_scaled)$predictions
p_rf_23 <- rf_raw_probs[, "2_3"]
p_rf_45 <- rf_raw_probs[, "4_5"]
denom_rf <- p_rf_23 + p_rf_45
denom_rf[denom_rf == 0] <- 1
p_router_23 <- p_rf_23 / denom_rf
p_router_45 <- p_rf_45 / denom_rf
# 3. Model 3A & 3B: Specialist Conditional Probabilities
all_feats <- c(binary_cols, cont_cols)
test_ds_scaled <- predict(art_ds$preproc, test_df)
test_ds_x      <- as.matrix(test_ds_scaled[, all_feats])
# P(ESI 2 | ESI 2 or 3)
if (!is.null(art_lgb23) && art_lgb23$has_lgb) {
  p_esi2_given_23 <- predict(art_lgb23$model, test_ds_x)
} else if (art_ds$has_lgb) {
  p_esi2_given_23 <- predict(art_ds$model_lgb, test_ds_x)
} else {
  p_esi2_given_23 <- predict(art_ds$model_lgb, xgb.DMatrix(data = test_ds_x))
}
p_esi3_given_23 <- 1 - p_esi2_given_23
# P(ESI 4 | ESI 4 or 5)
if (!is.null(art_xgb45)) {
  p_esi4_given_45 <- predict(art_xgb45$model, xgb.DMatrix(data = test_ds_x))
} else {
  p_esi4_given_45 <- predict(art_ds$model_xgb, xgb.DMatrix(data = test_ds_x))
}
p_esi5_given_45 <- 1 - p_esi4_given_45
# 4. PROBABILITY CHAIN RULE FOR ALL 5 CLASSES (1, 2, 3, 4, 5)
p_final_1 <- p_esi1
p_final_2 <- p_not_esi1 * p_router_23 * p_esi2_given_23
p_final_3 <- p_not_esi1 * p_router_23 * p_esi3_given_23
p_final_4 <- p_not_esi1 * p_router_45 * p_esi4_given_45
p_final_5 <- p_not_esi1 * p_router_45 * p_esi5_given_45
probs_5class_mat <- cbind(p_final_1, p_final_2, p_final_3, p_final_4, p_final_5)
colnames(probs_5class_mat) <- c("1", "2", "3", "4", "5")
# Verification of Total Probability Sum = 1.0 per sample
row_sums <- rowSums(probs_5class_mat)
cat(sprintf("5-Class Joint Probabilities Calculated for %d samples.\n", nrow(probs_5class_mat)))
cat(sprintf("Probability Sum Check across all test samples: Min = %.6f, Max = %.6f, Mean = %.6f\n",
            min(row_sums), max(row_sums), mean(row_sums)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Full 5-Class Holdout Test Benchmark (ESI 1 to 5)
# ---------------------------------------------------------
pred_idx <- apply(probs_5class_mat, 1, which.max)
pred_fac <- factor(colnames(probs_5class_mat)[pred_idx], levels = c("1", "2", "3", "4", "5"))
act_fac  <- factor(test_df$target_col, levels = c("1", "2", "3", "4", "5"))
cm_5class  <- confusionMatrix(pred_fac, act_fac)
acc_5class <- as.numeric(cm_5class$overall["Accuracy"])
prec_by_class    <- as.numeric(cm_5class$byClass[, "Pos Pred Value"])
rec_by_class     <- as.numeric(cm_5class$byClass[, "Sensitivity"])
spec_by_class    <- as.numeric(cm_5class$byClass[, "Specificity"])
bal_acc_by_class <- as.numeric(cm_5class$byClass[, "Balanced Accuracy"])
prec_by_class[is.na(prec_by_class)]       <- 0
rec_by_class[is.na(rec_by_class)]         <- 0
spec_by_class[is.na(spec_by_class)]       <- 0
bal_acc_by_class[is.na(bal_acc_by_class)] <- 0
f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 
                      2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
roc_auc_by_class <- sapply(1:5, function(i) {
  cls_name <- levels(act_fac)[i]
  act_bin  <- ifelse(act_fac == cls_name, 1, 0)
  r_obj    <- tryCatch(pROC::roc(act_bin, probs_5class_mat[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
mcc_by_class <- sapply(1:5, function(i) {
  cls <- levels(act_fac)[i]
  tp  <- sum(pred_fac == cls & act_fac == cls)
  tn  <- sum(pred_fac != cls & act_fac != cls)
  fp  <- sum(pred_fac == cls & act_fac != cls)
  fn  <- sum(pred_fac != cls & act_fac == cls)
  
  num   <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
})
actual_counts <- as.numeric(table(act_fac))
pred_counts   <- as.numeric(table(pred_fac))
diff_vec      <- pred_counts - actual_counts
diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
test_report_df <- data.frame(
  Class             = levels(act_fac),
  Actual_Count      = actual_counts,
  Pred_Count        = pred_counts,
  Diff              = diff_str,
  Precision         = round(prec_by_class, 4),
  Recall_Sens       = round(rec_by_class, 4),
  Specificity       = round(spec_by_class, 4),
  Balanced_Accuracy = round(bal_acc_by_class, 4),
  F1_Score          = round(f1_by_class, 4),
  ROC_AUC           = round(roc_auc_by_class, 4),
  MCC_Score         = round(mcc_by_class, 4)
)
macro_prec    <- mean(prec_by_class)
macro_rec     <- mean(rec_by_class)
macro_spec    <- mean(spec_by_class)
macro_bal_acc <- mean(bal_acc_by_class)
macro_f1      <- mean(f1_by_class)
macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
macro_mcc     <- mean(mcc_by_class)
cat(sprintf("============================================================\n"))
cat(sprintf("   5-CLASS PROBABILISTIC CHAIN RULE PIPELINE - HOLDOUT TEST BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Overall 5-Class Test Accuracy : %.4f (%.2f%%)\n", acc_5class, acc_5class * 100))
cat(sprintf("  Macro Balanced Accuracy       : %.4f\n", macro_bal_acc))
cat(sprintf("  Macro Specificity             : %.4f\n", macro_spec))
cat(sprintf("  Macro Precision               : %.4f\n", macro_prec))
cat(sprintf("  Macro Recall (Sensitivity)    : %.4f\n", macro_rec))
cat(sprintf("  Macro F1-Score                : %.4f\n", macro_f1))
cat(sprintf("  Macro ROC-AUC                 : %.4f\n", macro_roc_auc))
cat(sprintf("  Macro MCC Score               : %.4f\n", macro_mcc))
cat(sprintf("============================================================\n\n"))
cat("Holdout Test Set Per-Class Metrics (ESI 1, 2, 3, 4, 5):\n")
print(test_report_df)
cat("\nHoldout Test Set 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm_5class$table)
cat(sprintf("============================================================\n\n"))
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(test_report_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("5-Class Combined Pipeline Test Set Report written to: reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (5-Class Pipeline Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("Overall_Acc", "Macro_BalAcc", "Macro_Spec", "Macro_Prec", "Macro_Rec", "Macro_F1", "Macro_AUC", "Macro_MCC"),
                  levels = c("Overall_Acc", "Macro_BalAcc", "Macro_Spec", "Macro_Prec", "Macro_Rec", "Macro_F1", "Macro_AUC", "Macro_MCC")),
  Score  = c(acc_5class, macro_bal_acc, macro_spec, macro_prec, macro_rec, macro_f1, macro_roc_auc, macro_mcc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.3, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "5-Class Probabilistic Chain Rule Pipeline Benchmark (ESI 1 to 5)",
       subtitle = "Full System Test Benchmark (XGBoost ESI 1 + RF/XGB Router + LightGBM 2v3 + XGBoost 4v5)",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 9.5, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")
ggsave(file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p_bar, width = 11, height = 5, dpi = 300)
cat("5-Class Combined Pipeline Bar Chart saved to: plots/combined_pipeline_metrics_barchart.png\n")
p_bar